# Important functions and data

In [3]:
import nltk
from os import getcwd
import w1_unittest

nltk.download('twitter_samples')
nltk.download('stopwords')

[nltk_data] Downloading package twitter_samples to
[nltk_data]     /Users/bishanbhandari/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/bishanbhandari/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
filepath=f"{getcwd()}/../tmp2/"
nltk.data.path.append(filepath)
print(filepath)

/Users/bishanbhandari/anaconda_projects/db/SentimentalAnalysis/../tmp2/


In [5]:
import numpy as np
import pandas as pd
from nltk.corpus import twitter_samples

from utils import process_tweet, build_freqs

In [6]:
# select the set of positive and negative tweets
all_positive_tweets=twitter_samples.strings('positive_tweets.json')
all_negative_tweets=twitter_samples.strings('negative_tweets.json')

In [7]:
# splitting the data,one for training and another for testing 
test_pos=all_positive_tweets[4000:]
train_pos=all_positive_tweets[:4000]
test_neg=all_negative_tweets[4000:]
train_neg=all_negative_tweets[:4000]

train_x=train_pos+train_neg
test_x=test_pos+test_neg

In [8]:
# Combine positive and negative labels
train_y=np.append(np.ones((len(train_pos),1)),np.zeros((len(train_neg),1)),axis=0)
test_y=np.append(np.ones((len(test_pos),1)),np.zeros((len(test_neg),1)),axis=0)

In [9]:
# print the shape train and test sets
print("train_y.shape="+str(train_y.shape))
print("test_y="+str(test_y.shape))

train_y.shape=(8000, 1)
test_y=(2000, 1)


In [10]:
# create frequency dictionary
freqs=build_freqs(train_x,train_y)

# Check the output
print("type(freqs=)="+str(type(freqs)))
print("len(freqs)="+str(len(freqs.keys())))

type(freqs=)=<class 'dict'>
len(freqs)=11397


In [11]:
print('This is an example of a positive tweet:\n',train_x[0])
print('\nThis is an example of the processed version of the tweet:\n',process_tweet(train_x[0]))

This is an example of a positive tweet:
 #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)

This is an example of the processed version of the tweet:
 ['followfriday', 'top', 'engag', 'member', 'commun', 'week', ':)']


In [12]:
# UNQ_C1 GRADED FUNCTION:sigmoid
def sigmoid(z):
    '''
     Input:
        z: is the input (can be a scalar or an array)
    Output:
        h: the sigmoid of z
    '''

    h=None
    h=1/(1+np.exp(-z))

    return h
      

In [13]:
# Testing  function
if(sigmoid(0)==0.5):
    print("SUCCESS")
else:
    print("Oops!")
if(sigmoid(4.92)==0.9927537604041685):
    print("CORRECT")
else:
    print("Oops again")

SUCCESS
CORRECT


In [14]:
w1_unittest.test_sigmoid(sigmoid)

 All tests passed


In [15]:
# verify that when the model predicts close to 1,but the actual label is 0, the loss is a large positive value 
-1*(1-0)*np.log(1-0.9999)

9.210340371976294

In [16]:
# verify that when the model predicts close to 0 but the actual label is 1, the loss is a large positive value
-1 * np.log(0.0001) # loss is about 9.2

9.210340371976182

In [17]:
# UNQ_C2 GRADED FUNCTION: gradientDescent
def gradientDescent(x, y, theta, alpha, num_iters):
    '''
    Input:
        x: matrix of features which is (m,n+1)
        y: corresponding labels of the input matrix x, dimensions (m,1)
        theta: weight vector of dimension (n+1,1)
        alpha: learning rate
        num_iters: number of iterations you want to train your model for
    Output:
        J: the final cost
        theta: your final weight vector
    Hint: you might want to print the cost to make sure that it is going down.
    '''
    ### START CODE HERE ###
    # get 'm', the number of rows in matrix x
    m = None
    m = x.shape[0]
    
    for i in range(0, num_iters):
        
        # get z, the dot product of x and theta
        z = x.dot(theta) 
        
        #sigmoid of z
        h = sigmoid(z)
        
        #cost function
        J = -1/m*((y.T.dot(np.log(h))) + (1-y).T.dot(np.log(1-h)))

        # update the weights theta
        theta = theta - (alpha/m) *x.T.dot(h-y)
              
        
    ### END CODE HERE ###
    J = float(J)
    return J, theta

In [18]:
# validation of gradient descent
# Check the function
# Construct a synthetic test case using numpy PRNG functions
np.random.seed(1)
# X input is 10 x 3 with ones for the bias terms
tmp_X = np.append(np.ones((10, 1)), np.random.rand(10, 2) * 2000, axis=1)
# Y Labels are 10 x 1
tmp_Y = (np.random.rand(10, 1) > 0.35).astype(float)

# Apply gradient descent
tmp_J, tmp_theta = gradientDescent(tmp_X, tmp_Y, np.zeros((3, 1)), 1e-8, 700)
print(f"The cost after training is {tmp_J:.8f}.")
print(f"The resulting vector of weights is {[round(t, 8) for t in np.squeeze(tmp_theta)]}")

The cost after training is 0.67094970.
The resulting vector of weights is [4.1e-07, 0.00035658, 7.309e-05]


/var/folders/0f/5vzdhq951l30tm00tc0kgtgm0000gn/T/ipykernel_1117/3608199567.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


In [19]:
w1_unittest.test_gradientDescent(gradientDescent)

 All tests passed


/var/folders/0f/5vzdhq951l30tm00tc0kgtgm0000gn/T/ipykernel_1117/3608199567.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


In [20]:
# UNQ_C3 GRADED FUNCTION: extract_features
# process tweets and count frequencies
def extract_features(tweet, freqs, process_tweet=process_tweet):
    '''
    Input: 
        tweet: a list of words for one tweet
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
    Output: 
        x: a feature vector of dimension (1,3)
    '''
    # process_tweet tokenizes, stems, and removes stopwords
    word_l = process_tweet(tweet)
    
    # 3 elements in the form of a 1 x 3 vector
    x = np.zeros((1, 3)) 
    
    #bias term is set to 1
    x[0,0] = 1 
    
    
    # loop through each word in the list of words
    for word in word_l:
        
        j=(word,0)
        l=(word,1)
        
        # for -tive label 0
        if((word,0) in freqs):
            temp=freqs[j]
            x[0,2] = x[0,2]+temp
        
        #for the +tive label 1
        if((word,1) in freqs):
            temp1=freqs[l]
            x[0,1] =x[0,1]+temp1 


        
    ### END CODE HERE ###
    assert(x.shape == (1, 3))
    return x

In [21]:
# Check your function
# test 1 
# test on training data 
tmp1=extract_features(train_x[0],freqs)
print(tmp1)

[[1.000e+00 3.133e+03 6.100e+01]]


In [22]:
# Test 2:
# Check for when the words are not in the freqs dictionary
tmp2=extract_features('blorb bleeeeeb blooooob',freqs)
print(tmp2)

[[1. 0. 0.]]


In [23]:
# Test your function
w1_unittest.test_extract_features(extract_features,freqs)

 All tests passed


### Training Your Model
To train the model:

Stack the features for all training examples into a matrix X.
Call gradientDescent, which you've implemented above.

In [25]:
# collect the features 'x' and stack them into a matrix 'X'
X=np.zeros((len(train_x),3))
for i in range(len(train_x)):
    X[i,:]=extract_features(train_x[i],freqs)

# training labels corresponding to X
Y=train_y

# Apply gradient descent 
J,theta=gradientDescent(X,Y,np.zeros((3,1)),1e-9,1500)
print(f"The cost after training is{J:.8f}.")
print(f"The resulting vector of weights is {[round(t,8) for t in np.squeeze(theta)]}")

The cost after training is0.22524410.
The resulting vector of weights is [6e-08, 0.00053786, -0.00055885]


/var/folders/0f/5vzdhq951l30tm00tc0kgtgm0000gn/T/ipykernel_1117/3608199567.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


# Test logistic regression

It is time for you to test your logistic regression function on some new input that your model has not seen before.

Instructions: Write predict_tweet
Predict whether a tweet is positive or negative.

Given a tweet, process it, then extract the features.
Apply the model's learned weights on the features to get the logits.
Apply the sigmoid to the logits to get the prediction (a value between 0 and 1).

In [54]:
# UNQ_C4 GRADED FUNCTION: Predict_tweet 

def predict_tweet(tweet,freqs,theta):
    '''
    Input: 
        tweet: a string
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
        theta: (3,1) vector of weights
    Output: 
        y_pred: the probability of a tweet being positive or negative
    '''


    # extract the features of the tweet and store into x
    x=extract_features(tweet,freqs)

    # make the prediction using x and theta 
    y_pred=sigmoid(np.dot(x,theta))


    return y_pred

In [56]:
# Test function
for tweet in ['I am happy', 'I am bad', 'this movie should have been great.', 'great', 'great great', 'great great great', 'great great great great']:
    print('%s->%f'%(tweet,predict_tweet(tweet,freqs,theta)))

I am happy->0.519259
I am bad->0.494338
this movie should have been great.->0.515962
great->0.516052
great great->0.532070
great great great->0.548023
great great great great->0.563877


/var/folders/0f/5vzdhq951l30tm00tc0kgtgm0000gn/T/ipykernel_1117/2266168482.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('%s->%f'%(tweet,predict_tweet(tweet,freqs,theta)))


In [70]:
my_tweet='This is Bad'
print(predict_tweet(my_tweet,freqs,theta))

[[0.49433832]]


# Test Logistic Regression Performance

In [109]:
# UNQ_C5 GRADED FUNCTION: test_logistic_regression
def test_logistic_regression(test_x,test_y,freqs,theta,predict_tweet=predict_tweet):
     """
    Input: 
        test_x: a list of tweets
        test_y: (m, 1) vector with the corresponding labels for the list of tweets
        freqs: a dictionary with the frequency of each pair (or tuple)
        theta: weight vector of dimension (3, 1)
    Output: 
        accuracy: (# of tweets classified correctly) / (total # of tweets)
    """


    # the list for storing predictions 
     y_hat= []

     for tweet in test_x:
        # get the label prediction for the tweet 
        y_pred=predict_tweet(tweet,freqs,theta)

        if y_pred>0.5:
            y_hat.append(1.0)
        else:
            y_hat.append(0.0)

    # with the above implementation, y_hat is a list ,but test_y is (m,1) array
    # convert both to one-dimensional arrays in order to compare them using the '==' operator
     j=0
     y_hat_data=np.array(y_hat)
     test_y_data=test_y.flatten()

     for i in range(0,len(y_hat_data)):
        if y_hat_data[i]==test_y_data[i]:
            j+=1
     accuracy=j/len(test_y_data)
     accuracy=float(accuracy)

     return accuracy

In [111]:
tmp_accuracy = test_logistic_regression(test_x, test_y, freqs, theta)
print(f"Logistic regression model's accuracy = {tmp_accuracy:.4f}")

Logistic regression model's accuracy = 0.9965


In [113]:
# Test function 
w1_unittest.unittest_test_logistic_regression(test_logistic_regression,freqs,theta)

Wrong output type. 
	Expected: <class 'numpy.float64'>.
	Got: <class 'float'>.
Wrong output type. 
	Expected: <class 'numpy.float64'>.
	Got: <class 'float'>.
 2  Tests passed
 2  Tests failed


# Error Analysis

In [120]:
# Some error analysis
print('Label Predicted Tweet')
for x,y in zip(test_x,test_y):
    y_hat=predict_tweet(x,freqs,theta)

    if np.abs(y-(y_hat>0.5))>0:
        print('THE TWEET IS:',x)
        print('THE PROCESSED TWEET IS:',process_tweet(x))
        print('%d\t%0.8f\t%s'%(y,y_hat,''.join(process_tweet(x)).encode('ascii','ignore')))

Label Predicted Tweet
THE TWEET IS: @MarkBreech Not sure it would be good thing 4 my bottom daring 2 say 2 Miss B but Im gonna be so stubborn on mouth soaping ! #NotHavingit :p
THE PROCESSED TWEET IS: ['sure', 'would', 'good', 'thing', '4', 'bottom', 'dare', '2', 'say', '2', 'miss', 'b', 'im', 'gonna', 'stubborn', 'mouth', 'soap', 'nothavingit', ':p']
1	0.48927135	b'surewouldgoodthing4bottomdare2say2missbimgonnastubbornmouthsoapnothavingit:p'
THE TWEET IS: off to the park to get some sunlight : )
THE PROCESSED TWEET IS: ['park', 'get', 'sunlight']
1	0.49632427	b'parkgetsunlight'
THE TWEET IS: @msarosh Uff Itna Miss karhy thy ap :p
THE PROCESSED TWEET IS: ['uff', 'itna', 'miss', 'karhi', 'thi', 'ap', ':p']
1	0.48246190	b'uffitnamisskarhithiap:p'


/var/folders/0f/5vzdhq951l30tm00tc0kgtgm0000gn/T/ipykernel_1117/2155323405.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('%d\t%0.8f\t%s'%(y,y_hat,''.join(process_tweet(x)).encode('ascii','ignore')))


THE TWEET IS: @phenomyoutube u probs had more fun with david than me : (
THE PROCESSED TWEET IS: ['u', 'prob', 'fun', 'david']
0	0.50983758	b'uprobfundavid'
THE TWEET IS: pats jay : (
THE PROCESSED TWEET IS: ['pat', 'jay']
0	0.50040341	b'patjay'
THE TWEET IS: my beloved grandmother : ( https://t.co/wt4oXq5xCf
THE PROCESSED TWEET IS: ['belov', 'grandmoth']
0	0.50000001	b'belovgrandmoth'
THE TWEET IS: Sr. Financial Analyst - Expedia, Inc.: (#Bellevue, WA) http://t.co/ktknMhvwCI #Finance #ExpediaJobs #Job #Jobs #Hiring
THE PROCESSED TWEET IS: ['sr', 'financi', 'analyst', 'expedia', 'inc', 'bellevu', 'wa', 'financ', 'expediajob', 'job', 'job', 'hire']
0	0.50647821	b'srfinancianalystexpediaincbellevuwafinancexpediajobjobjobhire'


# Predict Own Tweet

In [154]:
my_tweet='watching FIFA Worldcup is so entertaining i absolutely love it '
print(process_tweet(my_tweet))
y_hat=predict_tweet(my_tweet,freqs,theta)
print(y_hat)
if y_hat>0.5:
    print('Positive Sentiment')
else:
    print('Negative Sentiment')

['watch', 'fifa', 'worldcup', 'entertain', 'absolut', 'love']
[[0.52385028]]
Positive Sentiment
